<a href="https://colab.research.google.com/github/missstechie/SDG-Classifier-MVP-/blob/main/Copy_of_SDG_Classifier(MVP).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers pandas scikit-learn

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [ ]:
sdgs = {
    "SDG 1": "No poverty",
    "SDG 2": "Zero hunger",
    "SDG 3": "Good health and well-being",
    "SDG 4": "Quality education",
    "SDG 5": "Gender equality",
    "SDG 6": "Clean water and sanitation",
    "SDG 7": "Affordable and clean energy",
    "SDG 8": "Decent work and economic growth",
    "SDG 9": "Industry, innovation and infrastructure",
    "SDG 10": "Reduced inequalities",
    "SDG 11": "Sustainable cities and communities",
    "SDG 12": "Responsible consumption and production",
    "SDG 13": "Climate action",
    "SDG 14": "Life below water",
    "SDG 15": "Life on land",
    "SDG 16": "Peace, justice and strong institutions",
    "SDG 17": "Partnerships for the goals"
}

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
sdg_texts = list(sdgs.values())
sdg_embeddings = model.encode(sdg_texts)

def _predict_sdg_with_details(text):
    text_embedding = model.encode(text)
    similarities = cosine_similarity([text_embedding], sdg_embeddings)[0]

    sdg_preds = []
    for i, sim in enumerate(similarities):
        sdg_code = list(sdgs.keys())[i]
        sdg_description = sdgs[sdg_code]
        sdg_preds.append((sdg_code, sdg_description, sim))

    # Sort by similarity in descending order and return top 3
    return sorted(sdg_preds, key=lambda x: x[2], reverse=True)[:3]

def predict_sdg(text):
    preds_with_details = _predict_sdg_with_details(text)
    if preds_with_details:
        top_pred = preds_with_details[0]
        return (top_pred[0], top_pred[1])
    return (None, None)

def predict_sdg_top3(text):
    preds_with_details = _predict_sdg_with_details(text)
    return [p[0] for p in preds_with_details]

def evaluate(df_to_evaluate):
    correct = 0
    total_predictions = len(df_to_evaluate)

    print(f"Evaluating {total_predictions} projects...")
    for _, row in df_to_evaluate.iterrows():
        pred_code, _ = predict_sdg(row["project"])

        if pred_code == row["actual_sdg"]:
            correct += 1

    accuracy = correct / total_predictions
    print(f"Accuracy: {accuracy:.4f} ({correct}/{total_predictions})\n")

In [ ]:
print(predict_sdg_top3("A mobile app that improves rural healthcare and education access"))

In [ ]:
test_text = "A mobile app that helps rural students access education"
print(predict_sdg(test_text))

In [ ]:
data = {
    "project": [
        "Mobile health app for villages",
        "Online education platform for students",
        "Clean drinking water system",
        "Solar energy for rural homes"
    ],
    "actual_sdg": [
        "SDG 3",
        "SDG 4",
        "SDG 6",
        "SDG 7"
    ]
}

df = pd.DataFrame(data)
df